In [1]:
import numpy as np
import time
from tokenizers import Tokenizer

In [ ]:
tokenizer_file = '/home/user/Desktop/ml/gpt/tokenizer/tokenizer-01.json'
tokenizer = Tokenizer.from_file(tokenizer_file)

current_step = 0
rng = np.random.default_rng(seed=42)

In [3]:
# evrything here can be changed each training session to optimize learning
batch_size = 32   # effective batch size is minibatch_size * accumulation_steps
block_size = 128    # max possible block size to train
learning_rate = 5e-4
save_iters = 50
eval_iters = 25

# everything below here NEEDS to be identical to load an extistng model
vocab_size = 30002
pad_id = 30000
assistant_id = 2
n_embed = 64
n_head = 4
n_layer = 8
dropout = 0.2

In [4]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()

tokens = np.array(tokenizer.encode(text, add_special_tokens=False).ids)
split_idx = int(len(tokens) * 0.9)
train_tokens = tokens[:split_idx]
val_tokens = tokens[split_idx:]

def create_batches(data_tokens, batch_size, block_size, rng):
    num_blocks = len(data_tokens) - block_size - 1
    usable_blocks = (num_blocks // batch_size) * batch_size
    indices = rng.permutation(np.arange(num_blocks))[:usable_blocks]
    return indices.reshape(-1, batch_size)

train_batches = create_batches(train_tokens, batch_size, block_size, rng)
val_batches = create_batches(val_tokens, batch_size, block_size, rng)

num_batches = train_batches.shape[0]
print(f"Train batches: {num_batches}")
print(f"Val batches: {val_batches.shape[0]}")

Train batches: 1617
Val batches: 176


In [5]:
def estimate_loss():
    out = {}
    for split in ['train', 'val']:
        out[split] = 0
        for k in range(eval_iters):
            X, Y = get_batch(split, batch_size)
            logits, loss, grad = m.forward(X, Y)
            out[split] += loss
        out[split] /= eval_iters
    return out

In [6]:
def get_batch(split, batch_size, step=None):
    global train_batches, val_batches, train_tokens, val_tokens
    if step is not None and step % num_batches == 0:
        train_batches = create_batches(train_tokens, batch_size, block_size, rng)
        val_batches = create_batches(val_tokens, batch_size, block_size, rng)
        
    data = train_tokens if split == 'train' else val_tokens
    batch_map = train_batches if split == 'train' else val_batches
    
    if step is None:
        ix = batch_map[rng.integers(batch_map.shape[0])]
    else:
        ix = batch_map[step % batch_map.shape[0]]

    x = np.vstack([data[i:i+block_size] for i in ix])
    y = np.vstack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [7]:
def sample_logits(logits, temp=1.0, top_k=0, top_p=0.0):
    logits /= temp

    if top_k > 0:
        k_th_val = np.partition(logits, -top_k, axis=-1)[..., -top_k]
        logits[logits < k_th_val[..., np.newaxis]] = -np.inf

    if top_p > 0.0:
        sorted_indices = np.argsort(logits, axis=-1)[..., ::-1]
        sorted_logits = np.take_along_axis(logits, sorted_indices, axis=-1)
        cumulative_probs = np.cumsum(softmax(sorted_logits), axis=-1)

        # Remove tokens with cumulative probability above the threshold
        sorted_indices_to_remove = cumulative_probs > top_p
        # Shift indices to keep the first token that exceeds top_p
        sorted_indices_to_remove = np.roll(sorted_indices_to_remove, 1, axis=-1)
        sorted_indices_to_remove[..., 0] = False

        mask = np.zeros_like(logits, dtype=bool)
        np.put_along_axis(mask, sorted_indices, sorted_indices_to_remove, axis=-1)
        logits[mask] = -np.inf

    probs = softmax(logits)
    next_tokens = np.array([np.random.choice(len(p), p=p) for p in probs])
    return next_tokens.reshape(-1, 1)

In [8]:
def softmax(logits):
    shift_logits = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(shift_logits)
    return exp / np.sum(exp, axis=-1, keepdims=True)

def cross_entropy(logits, targets):
    # Log-Sum-Exp trick for numerical stability
    max_logits = np.max(logits, axis=-1, keepdims=True)
    log_sum_exp = max_logits + np.log(np.sum(np.exp(logits - max_logits), axis=-1, keepdims=True))
    
    # Direct log-probability
    log_probs = logits - log_sum_exp
    
    # NLL Loss
    n_samples = logits.shape[0]
    loss = -np.sum(log_probs[np.arange(n_samples), targets]) / n_samples
    
    # Gradient calculation (this stays the same and is very stable)
    softmax_probs = np.exp(log_probs)
    target_array = np.zeros_like(logits)
    target_array[np.arange(n_samples), targets] = 1
    return loss, (softmax_probs - target_array) / n_samples
    
class AdamW:
    def __init__(self, params, betas=(0.9, 0.999), eps=1e-08, weight_decay=0.01):
        self.mt = np.zeros(params.shape)
        self.vt = np.zeros(params.shape)
        self.b1 = betas[0]
        self.b2 = betas[1]
        self.eps = eps
        self.wd = weight_decay
        self.t = 0

    def step(self, w, grad, L):
        self.t += 1
        w -= L * self.wd * w
        self.mt = self.b1 * self.mt + (1 - self.b1) * grad
        self.vt = self.b2 * self.vt + (1 - self.b2) * grad**2
        m_hat = self.mt / (1 - self.b1**self.t)
        v_hat = self.vt / (1 - self.b2**self.t)
        return w - L * m_hat / (np.sqrt(v_hat) + self.eps)
        
class ReLU:
    def __init__(self):
        self.z = None

    def forward(self, x):
        self.z = x
        return np.maximum(0, x)

    def backward(self, grad):
        return grad * (self.z > 0)

class GELU:
    def __init__(self):
        self.z = None
        self.tanh = None
        
    def forward(self, x):
        self.z = x
        self.tanh = np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3))
        return 0.5 * x * (1 + self.tanh)

    def backward(self, grad):
        return grad * 0.5 * (1 + self.tanh + self.z * np.sqrt(2/np.pi) * (1 - self.tanh**2) * (1 + 0.134145 * self.z**2))
        
class Dropout:
    def __init__(self, p):
        self.p = p
        self.is_training = True
        self.mask = None
        
    def forward(self, x):
        if self.is_training:
            self.mask = (np.random.rand(*x.shape) < (1 - self.p))
            return (x * self.mask) / (1 - self.p)
        else:
            self.mask = None
            return x
            
    def backward(self, grad):
        return (grad * self.mask) / (1 - self.p)
        
class RMSNorm:
    def __init__(self, dim):
        self.g = np.ones(dim)
        self.dg = np.zeros(dim)
        self.optim = AdamW(self.g, weight_decay=0)
        self.x = None
        
    def forward(self, x):
        self.x = x
        rms = np.sqrt(np.mean(x**2, axis=-1, keepdims=True) + 1e-7)
        return (x / rms) * self.g[np.newaxis, np.newaxis, :]
    
    def backward(self, grad):
        B, T, C = grad.shape
        x_flat = self.x.reshape(-1, C)
        grad_flat = grad.reshape(-1, C)
        
        rms = np.sqrt(np.mean(x_flat**2, axis=-1, keepdims=True) + 1e-7)
        x_hat = x_flat / rms
        
        self.dg += np.sum(x_hat * grad_flat, axis=0)
        
        grad_scaled = grad_flat * self.g
        dx_flat = (1.0 / rms) * (grad_scaled - x_hat * np.mean(grad_scaled * x_hat, axis=-1, keepdims=True))
        return dx_flat.reshape(B, T, C)
        
    def step(self, L):
        self.g = self.optim.step(self.g, self.dg, L)
    
    def zero_grad(self):
        self.dg = np.zeros(self.g.shape)

class Linear:
    def __init__(self, in_feat, out_feat, bias=True):
        self.w = np.random.normal(loc=0.0, scale=0.02, size=(in_feat, out_feat))
        self.dw = np.zeros(self.w.shape)
        self.woptim = AdamW(self.w)
        self.bias = bias
        if self.bias:
            self.b = np.zeros((1, out_feat))
            self.db = np.zeros(self.b.shape)
            self.boptim = AdamW(self.b, weight_decay=0)
        self.x = None

    def forward(self, x):
        self.x = x
        if self.bias:
            return (x @ self.w) + self.b
        else:
            return (x @ self.w)

    def backward(self, grad):
        B, T, C_in = self.x.shape
        x_flat = self.x.reshape(-1, C_in)
        grad_flat = grad.reshape(-1, grad.shape[-1])
        self.dw += x_flat.T @ grad_flat
        if self.bias:
            self.db += np.sum(grad_flat, axis=0, keepdims=True)
        dx_flat = grad_flat @ self.w.T
        return dx_flat.reshape(B, T, C_in)   # the gradients of the loss with respect to the input for this layer to be passed down

    def step(self, L):
        self.w = self.woptim.step(self.w, self.dw, L)
        if self.bias:
            self.b = self.boptim.step(self.b, self.db, L)

    def zero_grad(self):
        self.dw = np.zeros(self.w.shape)
        if self.bias:
            self.db = np.zeros(self.b.shape)

def rotate_half(x):
    x1, x2 = np.split(x, 2, axis=-1)
    return np.concatenate((-x2, x1), axis=-1)

def rotate_half_backward(x):
    x1, x2 = np.split(x, 2, axis=-1)
    return np.concatenate((x2, -x1), axis=-1)
    
class MultiHeadAttention:
    def __init__(self, n_embed, n_head, dropout):
        self.n_head = n_head
        self.n_embed = n_embed
        self.head_size = n_embed // n_head
        self.qkv_projection = Linear(n_embed, n_embed*3, bias=False)
        self.dropout1 = Dropout(dropout)
        self.proj = Linear(self.head_size * n_head, n_embed)
        self.dropout2 = Dropout(dropout)
        # torch.arange(0, D, 2) gets [0, 2, 4, ..., (D - 2)]
        self.inv_freq = 1.0 / (float(10000.0) ** (np.arange(0, self.head_size, 2).astype(float) / self.head_size))  # 1 / (base^(2i/D))
        t = np.arange(block_size*2)
        freqs = np.einsum("i,j->ij", t, self.inv_freq) # [T, head_size/2]
        emb = np.concatenate((freqs, freqs), axis=-1) # (T, head_size)
        self.cos = np.cos(emb)[np.newaxis, np.newaxis, :, :] # [1, 1, T, hs]
        self.sin = np.sin(emb)[np.newaxis, np.newaxis, :, :]

        self.cache = None

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_projection.forward(x)
        q, k, v = np.split(qkv, 3, axis=-1)  # (B, T, C)
        q = q.reshape(B, T, self.n_head, C // self.n_head).swapaxes(1, 2)  # (B, n_head, T, head_size)
        k = k.reshape(B, T, self.n_head, C // self.n_head).swapaxes(1, 2)
        v = v.reshape(B, T, self.n_head, C // self.n_head).swapaxes(1, 2)

        #print(f"({k[13][12][100][4]}, {k[13][12][100][(head_size//2)+4]})")
        #print(inv_freq[4])
        qr = (q * self.cos[:, :, :T, :]) + (rotate_half(q) * self.sin[:, :, :T, :])
        kr = (k * self.cos[:, :, :T, :]) + (rotate_half(k) * self.sin[:, :, :T, :])
        #print(f"({k[13][12][100][4]}, {k[13][12][100][(head_size//2)+4]})")
        
        qk = qr @ kr.swapaxes(-2, -1) * q.shape[-1]**-0.5
        qk += np.where(np.tril(np.ones((T, T))) == 0, -np.inf, 0)
        p = softmax(qk)
        w = self.dropout1.forward(p)
        out = w @ v

        self.cache = (x, q, k, v, qr, kr, p, w)
        
        out = out.swapaxes(1, 2).reshape(B, T, self.n_embed)
        out = self.dropout2.forward(self.proj.forward(out))
        return out

    def backward(self, grad):
        B, T, C = grad.shape
        x, q, k, v, qr, kr, p, w = self.cache
        grad = self.proj.backward(self.dropout2.backward(grad))
        grad = grad.reshape(B, T, self.n_head, C // self.n_head).swapaxes(1, 2)
        
        v_grad = w.swapaxes(-2, -1) @ grad
        grad = grad @ v.swapaxes(-2, -1)
        grad = self.dropout1.backward(grad)
        grad = p * (grad - np.sum(p * grad, axis=-1, keepdims=True))
        grad *= np.tril(np.ones((T, T)))
        
        q_grad = grad @ kr * q.shape[-1]**-0.5
        k_grad = grad.swapaxes(-2, -1) @ qr * q.shape[-1]**-0.5
        
        q_grad = (q_grad * self.cos[:, :, :T, :]) + rotate_half_backward(q_grad * self.sin[:, :, :T, :])
        k_grad = (k_grad * self.cos[:, :, :T, :]) + rotate_half_backward(k_grad * self.sin[:, :, :T, :])
        
        q_grad = q_grad.swapaxes(1, 2).reshape(B, T, self.n_embed)
        k_grad = k_grad.swapaxes(1, 2).reshape(B, T, self.n_embed)
        v_grad = v_grad.swapaxes(1, 2).reshape(B, T, self.n_embed)
        grad = np.concatenate((q_grad, k_grad, v_grad), axis=-1)
        return self.qkv_projection.backward(grad)
        
    def step(self, L):
        self.qkv_projection.step(L)
        self.proj.step(L)
        
    def zero_grad(self):
        self.qkv_projection.zero_grad()
        self.proj.zero_grad()
        
class FeedForward:
    def __init__(self, n_embed, dropout):
        self.linear1 = Linear(n_embed, 4 * n_embed)
        self.gelu = GELU()
        self.linear2 = Linear(4 * n_embed, n_embed)
        self.dropout = Dropout(dropout)

    def forward(self, x):
        x = self.gelu.forward(self.linear1.forward(x))
        return self.dropout.forward(self.linear2.forward(x))

    def backward(self, grad):
        grad = self.linear2.backward(self.dropout.backward(grad))
        return self.linear1.backward(self.gelu.backward(grad))
        
    def step(self, L):
        self.linear1.step(L)
        self.linear2.step(L)
        
    def zero_grad(self):        
        self.linear1.zero_grad()
        self.linear2.zero_grad()
        
class Block:
    def __init__(self, n_embed, n_head, dropout):
        self.sa = MultiHeadAttention(n_embed, n_head, dropout)
        self.ffwd = FeedForward(n_embed, dropout)
        self.ln1 = RMSNorm(n_embed)
        self.ln2 = RMSNorm(n_embed)

    def forward(self, x):
        x = x + self.sa.forward(self.ln1.forward(x))
        return x + self.ffwd.forward(self.ln2.forward(x))

    def backward(self, grad):
        grad = grad + self.ln2.backward(self.ffwd.backward(grad))
        return grad + self.ln1.backward(self.sa.backward(grad))
        
    def step(self, L):
        self.sa.step(L)
        self.ffwd.step(L)
        self.ln1.step(L)
        self.ln2.step(L)
        
    def zero_grad(self):
        self.sa.zero_grad()
        self.ffwd.zero_grad()
        self.ln1.zero_grad()
        self.ln2.zero_grad()
        
class Embedding:
    def __init__(self, vocab_size, n_embed):
        self.embeddings = np.random.normal(loc=0.0, scale=0.02, size=(vocab_size, n_embed))
        self.index = None
        self.de = np.zeros(self.embeddings.shape)
        self.optim = AdamW(self.embeddings)

    def forward(self, index):
        self.index = index
        return self.embeddings[index]

    def backward(self, grad):
        np.add.at(self.de, self.index, grad)
        
    def step(self, L):
        self.embeddings = self.optim.step(self.embeddings, self.de, L)
    
    def zero_grad(self):
        self.de = np.zeros(self.embeddings.shape)
        
class GPTLanguageModel:
    def __init__(self, vocab_size, n_embed, n_head, n_layer, dropout):
        self.token_embedding_table = Embedding(vocab_size, n_embed)
        self.blocks = [Block(n_embed, n_head, dropout) for _ in range(n_layer)]
        self.ln_f = RMSNorm(n_embed)
        self.lm_head = Linear(n_embed, vocab_size)
        
    def forward(self, index, targets=None):
        index = np.array(index)
        x = self.token_embedding_table.forward(index)
        
        for block in self.blocks:
            x = block.forward(x)
            
        x = self.ln_f.forward(x)
        logits = self.lm_head.forward(x)

        if targets is None:
            return logits
        else:
            B, T, C = logits.shape
            logits = logits.reshape(B*T, C)
            targets = np.array(targets).flatten()
            loss, grad = cross_entropy(logits, targets)
            return logits.reshape(B, T, C), loss, grad.reshape(B, T, C)

    def backward(self, grad):
        grad = self.lm_head.backward(grad)
        grad = self.ln_f.backward(grad)
        for block in reversed(self.blocks):
            grad = block.backward(grad)
        self.token_embedding_table.backward(grad)

    def step(self, L):
        self.lm_head.step(L)
        self.ln_f.step(L)
        for block in self.blocks:
            block.step(L)
        self.token_embedding_table.step(L)

    def zero_grad(self):
        self.lm_head.zero_grad()
        self.ln_f.zero_grad()
        for block in self.blocks:
            block.zero_grad()
        self.token_embedding_table.zero_grad()
        
    
    def generate(self, index, max_new_tokens, temp=1.0, top_k=0, top_p=0.0, eot_id=None):
        for _ in range(max_new_tokens):
            # get the logits for each char in the index
            logits = self.forward(index[..., -block_size:])
            # only look at the logits for each last letter
            logits = logits[:, -1, :]
            # choose one sample from our logits
            index_next = sample_logits(logits, temp, top_k, top_p)
            #concatenate the letter choice for each batch onto the end of the existing char list
            index = np.concatenate((index, index_next), axis=1) #(B, T+1)
            # stop generating if end of text token is generated
            if eot_id is not None and index_next.item() == eot_id:
                break
        return index

In [9]:
m = GPTLanguageModel(vocab_size, n_embed, n_head, n_layer, 0.0)

In [10]:
def load_weights(model, filename="np_model_weights_15000.npz"):
    data = np.load(filename)
    
    # Load Embeddings
    model.token_embedding_table.embeddings = data['tok_emb'].copy()
    
    for i, block in enumerate(model.blocks):
        # SA
        block.sa.qkv_projection.w = data[f'b{i}_qkv_w'].copy()
        block.sa.proj.w = data[f'b{i}_proj_w'].copy()
        block.sa.proj.b = data[f'b{i}_proj_b'].reshape(1, -1).copy()
        # Norms
        block.ln1.g = data[f'b{i}_ln1_g'].copy()
        block.ln2.g = data[f'b{i}_ln2_g'].copy()
        # FF
        block.ffwd.linear1.w = data[f'b{i}_ff_w1'].copy()
        block.ffwd.linear1.b = data[f'b{i}_ff_b1'].reshape(1, -1).copy()
        block.ffwd.linear2.w = data[f'b{i}_ff_w2'].copy()
        block.ffwd.linear2.b = data[f'b{i}_ff_b2'].reshape(1, -1).copy()

    # Final layers
    model.ln_f.g = data['ln_f_g'].copy()
    model.lm_head.w = data['head_w'].copy()
    model.lm_head.b = data['head_b'].reshape(1, -1).copy()
    print("NumPy model weights loaded and synchronized.")
load_weights(m)

NumPy model weights loaded and synchronized.


In [11]:
max_iters = 15000

In [12]:
losses = estimate_loss()    # estimate a base loss before training session
print(f"Step: {current_step:04d}, Train Loss: {losses['train']:.4f}, Val Loss: {losses['val']:.4f}")

times_tracker = {'train': {'time': 0.0, 'count': 0}, 'estimate': {'time': 0.0, 'count': 0}}
tic = time.perf_counter()
for step in range(current_step+1, max_iters + 1):
    m.zero_grad()
    xb, yb= get_batch('train', batch_size, step)
    logits, loss, grad = m.forward(xb, yb)
    m.backward(grad)
    
    m.step(learning_rate)
    current_step = step
    if step % save_iters == 0:
        times_tracker['train']['time'] += (time.perf_counter() - tic)
        times_tracker['train']['count'] += save_iters
        
        print()
        tic = time.perf_counter()
        losses = estimate_loss()
        times_tracker['estimate']['time'] += (time.perf_counter() - tic)
        times_tracker['estimate']['count'] += 1
        print(f"Step: {step:04d}, Train Loss: {losses['train']:.4f}, Val Loss: {losses['val']:.4f}")
        
        tic = time.perf_counter()

Step: 0000, Train Loss: 0.0761, Val Loss: 14.8768


KeyboardInterrupt: 

In [15]:
print(f"Total time train: {times_tracker['train']['time'] /60/60:.3f}hr")
print(f"Total time estim: {times_tracker['estimate']['time'] /60/60:.3f}hr")
print(f"Average time per Optimizer step: {times_tracker['train']['time'] / times_tracker['train']['count']:.3f}sec")
print(f"Average time {save_iters} Optimizr steps: {times_tracker['train']['time'] / (times_tracker['train']['count'] / save_iters)/60:.3f}min")
print(f"Average time per Estimate  Loss: {times_tracker['estimate']['time'] / times_tracker['estimate']['count']:.3f}sec")

Total time train: 11.625hr
Total time estim: 9.173hr
Average time per Optimizer step: 2.790sec
Average time 50 Optimizr steps: 2.325min
Average time per Estimate  Loss: 110.071sec


In [13]:
print(tokenizer.decode(m.generate(np.array([tokenizer.encode("I believe, my dears, that I am the proudest story-teller", add_special_tokens=False).ids]), 512, temp=0.8, top_k=50, top_p=0.9).tolist()[0]))

I believe, my dears, that I am the proudest story-teller that ever
lived. Many a time tears of pride and joy have stood in my eyes while I
read the tender, loving, appealing letters that come to me in almost
every mail from my little readers. To have pleased you, to have
interested you, to have won your friendship, and perhaps your love,
through my stories, is to my mind as great an achievement as to become
President of the United States. Indeed, I would much rather be your
story-teller, under these conditions, than to be the President. So you
have helped me to fulfill my life's ambition, and I am more grateful to
you, my dears, than I can express in words.

I try to answer every letter of my young correspondents; yet sometimes
there are so many letters that a little time must pass before you get
your answer. But be patient, friends, for the answer will surely come,
and by writing to me you more than repay me for the pleasant task of
preparing these books. Besides, I am proud to acknow

In [14]:
def save_np_to_numpy(model, filename="np_model_weights_15000.npz"):
    weights_dict = {}
    
    # Embeddings
    weights_dict['tok_emb'] = model.token_embedding_table.embeddings
    
    # Blocks
    for i, block in enumerate(model.blocks):
        # Attention - Transpose Linear weights for your (x @ W) format
        weights_dict[f'b{i}_qkv_w'] = block.sa.qkv_projection.w
        weights_dict[f'b{i}_proj_w'] = block.sa.proj.w
        weights_dict[f'b{i}_proj_b'] = block.sa.proj.b
        
        # RMSNorm
        weights_dict[f'b{i}_ln1_g'] = block.ln1.g
        weights_dict[f'b{i}_ln2_g'] = block.ln2.g
        
        # MLP (FeedForward)
        weights_dict[f'b{i}_ff_w1'] = block.ffwd.linear1.w
        weights_dict[f'b{i}_ff_b1'] = block.ffwd.linear1.b
        weights_dict[f'b{i}_ff_w2'] = block.ffwd.linear2.w
        weights_dict[f'b{i}_ff_b2'] = block.ffwd.linear2.b

    # Head and Final Norm
    weights_dict['ln_f_g'] =  model.ln_f.g
    weights_dict['head_w'] =  model.lm_head.w
    weights_dict['head_b'] =  model.lm_head.b

    np.savez(filename, **weights_dict)
    print(f"Weights saved to {filename}")
save_np_to_numpy(m)

Weights saved to np_model_weights_15000.npz


In [11]:
def check_np_gradients(model, step):
    print(f"\n=== NumPy Gradient Check {step} ===")
    
    # 1. LM Head (The "Entry" Point for Backprop)
    print(f"LM Head dW Max: {np.max(np.abs(model.lm_head.dw)):.8e}")
    print(f"LM Head db Max: {np.max(np.abs(model.lm_head.db)):.8e}")
    
    # 2. Final Norm
    print(f"Final LN dg Max: {np.max(np.abs(model.ln_f.dg)):.8e}")
    
    # 3. FeedForward (Inside the Block)
    ff = model.blocks[-1].ffwd
    print(f"FF Linear2 dW Max: {np.max(np.abs(ff.linear2.dw)):.8e}")
    print(f"FF Linear2 db Max: {np.max(np.abs(ff.linear2.db)):.8e}")
    print(f"FF Linear1 dW Max: {np.max(np.abs(ff.linear1.dw)):.8e}")
    print(f"FF Linear1 db Max: {np.max(np.abs(ff.linear1.db)):.8e}")

    print(f"LN2 (FF) dg Max: {np.max(np.abs(model.blocks[-1].ln2.dg)):.8e}")
    
    # 4. Attention (Inside the Block)
    sa = model.blocks[-1].sa
    print(f"SA Proj dW Max: {np.max(np.abs(sa.proj.dw)):.8e}")
    print(f"SA Proj db Max: {np.max(np.abs(sa.proj.db)):.8e}")
    print(f"SA QKV  dW Max: {np.max(np.abs(sa.qkv_projection.dw)):.8e}")

    print(f"LN1 (SA) dg Max: {np.max(np.abs(model.blocks[-1].ln1.dg)):.8e}")

In [12]:
losses = estimate_loss()    # estimate a base loss before training session
print(f"Step: {current_step:04d}, Train Loss: {losses['train']:.4f}, Val Loss: {losses['val']:.4f}")

for step in range(current_step+1, 2 + 1):
    m.zero_grad()
    xb, yb= get_batch('train', batch_size, step)
    logits, loss, grad = m.forward(xb, yb)
    print(f"\nLoss: {loss}")
    m.backward(grad)
    check_np_gradients(m, step)
    m.step(learning_rate)

Step: 0000, Train Loss: 10.3169, Val Loss: 10.3167

Loss: 10.315444124008295

=== NumPy Gradient Check 1 ===
LM Head dW Max: 3.50644678e-02
LM Head db Max: 4.56220665e-02
Final LN dg Max: 1.97043018e-03
FF Linear2 dW Max: 5.60609729e-03
FF Linear2 db Max: 6.80883285e-02
FF Linear1 dW Max: 5.25193626e-03
FF Linear1 db Max: 6.93722721e-03
LN2 (FF) dg Max: 6.01764638e-04
SA Proj dW Max: 6.51904063e-03
SA Proj db Max: 6.07210050e-02
SA QKV  dW Max: 6.85251173e-03
LN1 (SA) dg Max: 5.71049822e-04

Loss: 10.236489787954387

=== NumPy Gradient Check 2 ===
LM Head dW Max: 7.54186819e-02
LM Head db Max: 4.51248499e-02
Final LN dg Max: 7.64627781e-03
FF Linear2 dW Max: 7.75655653e-03
FF Linear2 db Max: 3.85947475e-02
FF Linear1 dW Max: 6.49478583e-03
FF Linear1 db Max: 3.96079568e-03
LN2 (FF) dg Max: 1.01984771e-03
SA Proj dW Max: 1.05217270e-02
SA Proj db Max: 3.69727838e-02
SA QKV  dW Max: 1.01475645e-02
LN1 (SA) dg Max: 1.24831982e-03
